# Exploring the Data Lake

**Three zoom levels:**

| Level | Question | Tool | Cost |
|---|---|---|---|
| **Storage** | What files physically exist? | `s3_ls()` (boto3) | free |
| **Catalog** | What's registered as a queryable table? | `databases()`, `tables()`, `describe()` (Glue) | free |
| **Data** | What do the values look like? | `aq()` (Athena) or `dread()` (DuckDB) | small / local |

Note: If anything says *"token expired"*, run `aws sso login --profile ciccada` in a terminal and re-run the cell.

In [1]:
from aws_config import *
# confirm you're logged in
# if not, run "aws sso login --profile ciccada" in your terminal and try again
session.client('sts').get_caller_identity()['Arn']

'arn:aws:sts::130340360668:assumed-role/AWSReservedSSO_AWSAdministratorAccess_58ece215f84a4b54/z3553082_sa@ad.unsw.edu.au'

## Imports & declarations

In [2]:
import pandas as pd
# Show every single column and avoid truncating wide text cells
pd.set_option('display.max_columns', None)
# On and OFF to check results within tables
## This breaks it!!! too many rows to display, so only use when you need to see all rows
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_colwidth', None)

import pytz

## [Level 1] Storage: the physical files in S3

`s3_ls()` lists what's actually in the bucket. 

This is ground truth. Works even for data that isn't in Glue yet.

In [3]:
# Top level of the bucket. Major data areas
s3_ls()

,type,name,size_mb
0,folder,BOM_NCI/,None
1,folder,DeepArchive/,None
2,folder,Flask_App/,None
3,folder,PostgreJDBC_Driver/,None
4,folder,SAPN/,None
5,folder,SAPNTest/,None
6,folder,Trino-Warehouse/,None
7,folder,athena-results/,None
8,folder,spark-warehouse/,None
9,folder,temp_SolA/,None


In [4]:
# Inside the Spark warehouse is where results tables seem to be written
s3_ls('spark-warehouse/')

,type,name,size_mb
0,folder,spark-warehouse/Compliance_results_SolA/,None
1,folder,spark-warehouse/SolA_metadata/,None
2,folder,spark-warehouse/bucketed_table4_partitions_loo...,None


In [5]:
# Drill into one table's folder to see its partitioning.
# Partition folders look like year=2025/month=1/
# This is what makes `WHERE year=2025 AND month=1` cheap (Athena reads only those folders).
s3_ls('Trino-Warehouse/')

,type,name,size_mb
0,folder,Trino-Warehouse/BOM_NCI/,None
1,folder,Trino-Warehouse/SAPN2022/,None
2,folder,Trino-Warehouse/solar_analytics/,None


In [6]:
s3_ls('Trino-Warehouse/solar_analytics/')

,type,name,size_mb
0,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
1,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
2,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
3,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
4,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
...,...,...,...
59,folder,Trino-Warehouse/solar_analytics/test_sola_2025...,None
60,folder,Trino-Warehouse/solar_analytics/ts-957d79213e8...,None
61,folder,Trino-Warehouse/solar_analytics/voltwatt_uncur...,None
62,folder,Trino-Warehouse/solar_analytics/voltwatt_uncur...,None


## [Level 2] Catalog: what's queryable, and its schema

Glue is the index that lets us write SQL. These calls are (metadata only).

In [8]:
databases()

,Database,Description
0,bom_nci,
1,default,Default Hive database
2,elb_logdb,
3,sapn2022,
4,solar_analytics,Migrated from Hive Metastore
5,solar_analytics_iceberg,
6,test_db,
7,type_probe,


In [11]:
# Every table in every database, in one view.
all_tabs = pd.concat([tables(d) for d in databases()['Database']], ignore_index=True)
all_tabs[['Database', 'Table']]

,Database,Table
0,bom_nci,solar
1,elb_logdb,elb_logs_tbl
2,sapn2022,circuit_measurements
3,sapn2022,circuit_measurements_curtailment_train
4,solar_analytics,circuits
5,solar_analytics,compliance_voltvar
6,solar_analytics,compliance_voltwatt
7,solar_analytics,meta_single_inverters
8,solar_analytics,meta_single_inverters_wrong_capacity
9,solar_analytics,meta_single_inverters_wrong_capacity_up2_3c


In [12]:
# The columns + types of any table. 
# Works for Iceberg tables too (where the Glue column list is often blank). 
# Kind of 'data dictionary' lookup.
table = "conformance_voltvar_v2"
database = "solar_analytics_iceberg"
aq(f"SELECT * FROM {table} LIMIT 5", database=database)

,site_id,day,day_night,p_kw_sum,nonconformance_voltvar_sum,q_adverse_sum,q_inactive_sum,q_significant_shortfall_sum,q_near_conformant_sum,q_major_surplus_sum,curtailment_voltvar_sum,nonconformance_voltvar_red_sum,nonconformance_voltvar_red_alt_sum,nonconformance_voltvar_count,q_adverse_count,q_inactive_count,q_significant_shortfall_count,q_near_conformant_count,q_major_surplus_count,curtailment_voltvar_count,nonconformance_voltvar_red_count,nonconformance_voltvar_red_alt_count,curtailment_eligible_count,null_uncurtailed_p_count,exposed_count,all_intervals_count,total_count,year,month
0,1203795895,15,day,253.105608,7.639319,0.000000,7.639319,0.0,0.0,0.0,NaN,7.639319,7.639319,2,0,2,0,0,0,0,2,2,0,0,209,209,209,2025,5
1,490849991,27,night,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,NaN,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0,0,60,144,144,2025,5
2,1382273135,29,day,547.791431,0.293151,0.293151,0.000000,0.0,0.0,0.0,NaN,0.293151,0.293151,11,11,0,0,0,0,0,11,11,0,0,53,218,218,2025,5
3,1652679703,9,day,349.508450,141.612977,0.249239,141.363738,0.0,0.0,0.0,NaN,141.612977,141.612977,151,3,148,0,0,0,0,151,151,0,0,202,202,202,2025,5
4,124828623,25,night,-0.722694,0.048031,0.048031,0.000000,0.0,0.0,0.0,NaN,0.048031,0.048031,2,2,0,0,0,0,0,2,2,0,0,68,143,143,2025,5


## [Level 3] Data peek at actual rows

Two engines, same files. 

1. Use `aq()` (Athena) for normal SQL
2. use `dread()` (DuckDB, reads the Parquet file directly) for quick peeks or when Glue's description is wrong.

**Cost:** on the big `ts` table, always filter on `year`/`month`/`is_pv` and never sort on a fresh peek.

In [ ]:
# Dimension tables are tiny
aq('SELECT * FROM circuits LIMIT 10')

In [ ]:
# The telemetry fact table: a CHEAP peek (no ORDER BY, just LIMIT).
# This shows the real column names so we can write proper filtered queries next.
aq("SELECT * FROM ts WHERE is_pv = True LIMIT 5", database='solar_analytics_iceberg')

In [ ]:
# How many rows in one month of PV telemetry? (count scans only the partition)
aq('''
    SELECT count(*) AS rows
    FROM ts
    WHERE is_pv = True AND year = 2025 AND month = 1
''', database='solar_analytics_iceberg')

## If Athena chokes: read the Parquet directly with DuckDB

Some results tables have *schema drift*. The Glue description disagrees with the file (e.g. a column the file stores as integer but Glue calls a double). This is common.

Athena won't be able to read it, so DuckDB reads the file's own schema and just works.

The S3 path comes from the error message, or from `s3_ls()`.

In [ ]:
dread('s3://project-ciccada/spark-warehouse/'
      'Compliance_results_SolA/compliance_voltvar.parquet/*.parquet')

## Recipe to explore any new data source in future

1. **`s3_ls('<prefix>/')`**: See what physically exists and how it's "foldered".
2. **`databases()` / `tables(db)`**: Check if it is registered in Glue. If yes, can use SQL.
3. **`describe('<table>', db)`**: Learn its columns.
4. **`aq('SELECT ... LIMIT 5')`**: Peek at values (filter on partitions if it's big).
5. Note: Not in Glue, or Glue is wrong?: Point **`dread('s3://.../*.parquet')`** straight at the files.

# Exploratory Data Analysis - Solar Analytics

In [15]:
# =============================================================================
# Exploratory Data Analysis Solar Analytics
# Goal: understand the time range, site/circuit counts, and data shape before touching the analysis notebooks.
# =============================================================================

# Short aliases for the two databases we'll use constantly
SA  = "solar_analytics"
SAI = "solar_analytics_iceberg"

## Sites and circuits

In [ ]:
# -----------------------------------------------------------------------------
# Assumptions:
# sites  = physical locations (one row per address)
# circuits = monitoring points within a site (one row per phase/device)
# is_pv = True means it's a solar PV circuit (not load, battery, etc.)
# -----------------------------------------------------------------------------

site_count = aq("SELECT count(*) AS n_sites FROM sites", database=SAI)
print("Total sites:", site_count["n_sites"].iloc[0])

In [ ]:
circuit_counts = aq("""
    SELECT
        is_pv,
        count(*)          AS n_circuits,
        count(DISTINCT site_id) AS n_sites
    FROM circuits
    GROUP BY is_pv
    ORDER BY is_pv DESC
""", 
database=SAI)
circuit_counts

# is_pv=True rows are what the telemetry analysis is built on.
# The values below double count sites because many sites have both PV and non-PV circuits.

In [ ]:
# How many circuits per site?
circuits_per_site = aq("""
    SELECT
        circuit_count,
        count(*) AS n_sites
    FROM (
        SELECT site_id, count(*) AS circuit_count
        FROM circuits
        WHERE is_pv = True
        GROUP BY site_id
    )
    GROUP BY circuit_count
    ORDER BY circuit_count
""", database=SAI)
circuits_per_site

## Geo spread

In [ ]:
# -----------------------------------------------------------------------------
# Geographic spread — use meta_up23c, not sites
# -----------------------------------------------------------------------------

states = aq("""
    SELECT state, count(*) AS n_sites
    FROM meta_up23c
    GROUP BY state
    ORDER BY n_sites DESC
""", database=SAI)
states

In [ ]:
# -----------------------------------------------------------------------------
# Use partition_lookup (tiny table) rather than querying ts directly.
# Reading min/max timestamps from a billions-row table is expensive;
# the lookup table gives you the answer for free.
# -----------------------------------------------------------------------------

partitions = aq("SELECT * FROM partition_lookup ORDER BY year, month", database=SA)
partitions
# Each row = one (year, month) partition that exists in the ts table.
# The first and last rows tell you the data window.

In [ ]:
unique_months = partitions[['year', 'month']].drop_duplicates().sort_values(['year', 'month'])

print("Data covers:")
print(f"  From: {unique_months['year'].iloc[0]}-{str(unique_months['month'].iloc[0]).zfill(2)}")
print(f"  To:   {unique_months['year'].iloc[-1]}-{str(unique_months['month'].iloc[-1]).zfill(2)}")
print(f"  Total months: {len(unique_months)}")

## Export limits

In [ ]:
aq('''
SELECT flex_export_detected, count(DISTINCT site_id) AS n_sites
FROM meta_up23c
WHERE is_pv = True
GROUP BY flex_export_detected
''', database='solar_analytics_iceberg')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# flex_export_detected diagnostic checks
# Paste these cells into your notebook after the prevalence check
# ═══════════════════════════════════════════════════════════════════════════════

# %% ── Check 1: Where are the flagged sites? ─────────────────────────────────
# Break down by state and DNSP to see if flex export is concentrated
# (e.g. SA Power Networks runs a major flexible export program)

flex_by_dnsp = aq("""
    SELECT
        flex_export_detected,
        state,
        dnsp_name,
        count(DISTINCT site_id)  AS n_sites,
        round(avg(ac_capacity_kw), 1) AS avg_ac_kw,
        round(avg(export_limit_kw), 1) AS avg_export_limit_kw
    FROM meta_up23c
    WHERE is_pv = True
    GROUP BY flex_export_detected, state, dnsp_name
    ORDER BY flex_export_detected DESC, n_sites DESC
""", database='solar_analytics_iceberg')

print("Flex-export sites by state and DNSP:")
print(flex_by_dnsp.to_string(index=False))

# %% ── Check 2: Do flagged sites have explicit export limits? ─────────────────
# If export_limit_kw is set AND is less than ac_capacity_kw, that's a strong
# signal the site really is DOE-constrained.

flex_export_limits = aq("""
    SELECT
        flex_export_detected,
        count(DISTINCT site_id) AS n_sites,
        sum(CASE WHEN export_limit_kw IS NOT NULL THEN 1 ELSE 0 END) AS has_export_limit,
        sum(CASE WHEN export_limit_kw IS NOT NULL
                  AND export_limit_kw < ac_capacity_kw THEN 1 ELSE 0 END)
            AS export_limit_below_nameplate
    FROM (
        SELECT DISTINCT site_id, ac_capacity_kw, export_limit_kw, flex_export_detected
        FROM meta_up23c
        WHERE is_pv = True
    )
    GROUP BY flex_export_detected
""", database='solar_analytics_iceberg')

print("\nExport limit breakdown:")
print(flex_export_limits.to_string(index=False))

## Schema of the key tables

In [ ]:
# Schema discovery for all Iceberg tables
# DESCRIBE doesn't work for Iceberg via Glue.
# Column names come back as DataFrame headers from SELECT * LIMIT 1.

iceberg_tables = [
    "ts", 
    "circuits",
    "sites", 
    "conformance_voltwatt",
    "conformance_voltvar", 
    "conformance_antiisland",
    "conformance_sust_op", 
    "all_uncurtailedpv", 
    "meta_up23c"
]

schemas = {}
for t in iceberg_tables:
    try:
        row = aq(f"SELECT * FROM {t} LIMIT 1", database=SAI)
        schemas[t] = pd.DataFrame({
            "column": row.columns.tolist(),
            "dtype":  [str(d) for d in row.dtypes.tolist()]
        })
        print(f"✓ {t:35s} {len(row.columns)} columns")
    except Exception as e:
        print(f"✗ {t:35s} ERROR: {e}")

In [ ]:
# Open up schemas here:
schemas['ts']

In [ ]:
# Open up schemas here:
schemas['meta_up23c']

In [ ]:
test = aq("""
SELECT t_stamp, voltage, power
FROM ts
WHERE year = 2025 AND month = 5 AND is_pv = True
  AND circuit_id = 291493
ORDER BY t_stamp
LIMIT 288
   """, database=SAI)
test

In [ ]:
test.to_csv('test.csv', index=False)

In [ ]:
# =============================================================================
# The metadata table is meta_up23c, not sites
# sites (Iceberg) only has 3 columns; all rich metadata is in meta_up23c
# =============================================================================

print("meta_up23c columns:")
print(schemas["meta_up23c"]["column"].tolist())

In [ ]:
# Circuit count banded: makes the 1-to-60 range interpretable
circuit_profile = aq("""
    SELECT
        CASE
            WHEN circuit_count = 1  THEN '1  - single phase'
            WHEN circuit_count = 2  THEN '2  - split or data gap'
            WHEN circuit_count = 3  THEN '3  - three phase'
            WHEN circuit_count <= 6 THEN '4-6 - multi-array or battery'
            ELSE                         '7+  - commercial / large site'
        END AS circuit_profile,
        count(*) AS n_sites,
        round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM (
        SELECT site_id, count(*) AS circuit_count
        FROM circuits
        WHERE is_pv = True
        GROUP BY site_id
    )
    GROUP BY 1
    ORDER BY min(circuit_count)
""", database=SAI)
circuit_profile

In [ ]:
meta_count = aq("""
    SELECT count(*) AS n_sites,
           count(DISTINCT state) AS n_states
    FROM meta_single_inverters
""", database=SA)
meta_count

## Fleet characteristics

In [ ]:
# -----------------------------------------------------------------------------
# Site summary four separate queries, one per count
# (Athena can't mix two databases in a single query)
# -----------------------------------------------------------------------------

n_sites         = aq("SELECT count(DISTINCT site_id) AS n FROM sites",       database=SAI)["n"].iloc[0]
n_meta_up23c    = aq("SELECT count(DISTINCT site_id) AS n FROM meta_up23c",  database=SAI)["n"].iloc[0]
n_pv_circuits   = aq("SELECT count(DISTINCT site_id) AS n FROM circuits WHERE is_pv = True", database=SAI)["n"].iloc[0]
#n_single_inv    = aq("SELECT count(*) AS n FROM meta_single_inverters",       database=SA) ["n"].iloc[0]

site_summary = pd.DataFrame([{
    "total_in_sites_table":       n_sites,
    "sites_in_meta_up23c":        n_meta_up23c,
    "sites_with_pv_circuit":      n_pv_circuits
#    "meta_single_inverters_rows": n_single_inv,
}])
site_summary

# Telemetry plots: one month, one circuit

In [ ]:
# =============================================================================
# Telemetry explorer. Single site, one month, voltage + generation plot
#
# KEY CONCEPT: Select by SITE, not circuit.
#   - site_id  = the physical address (one row in meta_up23c)
#   - circuit_id = one monitoring channel per phase
#   - The conformance results are per site_id, so site is the selector.
#   - Then look up the circuit_id(s) for that site to pull raw telemetry.
# =============================================================================

## Definitions

In [16]:
# AS/NZS 4777.2:2020 Australia-A set-points + pipeline constants
AS4777 = {
    # Volt-Watt: 100% S_rated at/below V1, ramp to 20% at V2
    "VW":   {"V1": 253.0, "V2": 260.0, "P1": 1.00, "P2": 0.20},

    # Volt-VAr: +Q1 supplying <=V1, 0 across deadband V2..V3, -Q4 absorbing >=V4
    "VVAR": {"V1": 207.0, "V2": 220.0, "V3": 240.0, "V4": 258.0,
             "Q1": 0.44, "Q4": 0.60},

    # Volt-VAr Q_impact category thresholds (signed measured/required ratio)
    "QIMP": {"thr1": -0.1, "thr2": 0.1, "thr3": 0.9, "thr4": 1.1},

    "TOL_FRAC":         0.04,   # +/-4% S_rated tolerance band, used everywhere
    "SITE_CONF_THRESH": 0.10,   # site conformant if nonconf fraction <= 10%
    "PV_ACTIVE_FRAC":   0.16,   # 'solar generating' = P_pv > 16% S_rated
    "INTERVAL_H":       5/60,   # 5-min interval -> hours, for kW-sum -> kWh

    # Sustained-operation: inverter must cease generating if V sustained
    # above this threshold for 3 consecutive 5-min intervals (~15 min).
    # Source: AS/NZS 4777.2:2020 
    "SUSTOP_V_CEIL":    258.0,
    "SUSTOP_V_BAND":    (253, 258),  # sweep range stored in conformance_sust_op*

    # Anti-islanding thresholdsAS/NZS 4777.2:2020
    # Under-voltage (UV): voltage too low, inverter must disconnect
    # These sit far below normal operating range; shown as annotation not axis lines for now
    "AI_UV2":  70.0,    # V ≤ 70 V  → disconnect in ≤ 2 s
    "AI_UV1": 180.0,    # V ≤ 180 V → disconnect in ≤ 11 s
    # Over-voltage (OV): 
    "AI_OV1": 265.0,    # V ≥ 265 V → disconnect in ≤ 2 s
    "AI_OV2": 275.0,    # V ≥ 275 V → disconnect in ≤ 0.2 s
    # Note: at 5-min telemetry resolution these sub-second trips are not directly observable

    "AI_V_BAND":        (260, 265),  # sweep range stored in conformance_antiisland
}

print("AS4777 constants loaded:")
for k, v in AS4777.items():
    print(f"  {k:20s} = {v}")

AS4777 constants loaded:
  VW                   = {'V1': 253.0, 'V2': 260.0, 'P1': 1.0, 'P2': 0.2}
  VVAR                 = {'V1': 207.0, 'V2': 220.0, 'V3': 240.0, 'V4': 258.0, 'Q1': 0.44, 'Q4': 0.6}
  QIMP                 = {'thr1': -0.1, 'thr2': 0.1, 'thr3': 0.9, 'thr4': 1.1}
  TOL_FRAC             = 0.04
  SITE_CONF_THRESH     = 0.1
  PV_ACTIVE_FRAC       = 0.16
  INTERVAL_H           = 0.08333333333333333
  SUSTOP_V_CEIL        = 258.0
  SUSTOP_V_BAND        = (253, 258)
  AI_UV2               = 70.0
  AI_UV1               = 180.0
  AI_OV1               = 265.0
  AI_OV2               = 275.0
  AI_V_BAND            = (260, 265)


## Site selection

In [17]:
# Quick schema check
_vv_schema = aq("SELECT * FROM conformance_voltvar_v2 LIMIT 1", database="solar_analytics_iceberg")
print("conformance_voltvar columns:")
print(list(_vv_schema.columns))

conformance_voltvar columns:
['site_id', 'day', 'day_night', 'p_kw_sum', 'nonconformance_voltvar_sum', 'q_adverse_sum', 'q_inactive_sum', 'q_significant_shortfall_sum', 'q_near_conformant_sum', 'q_major_surplus_sum', 'curtailment_voltvar_sum', 'nonconformance_voltvar_red_sum', 'nonconformance_voltvar_red_alt_sum', 'nonconformance_voltvar_count', 'q_adverse_count', 'q_inactive_count', 'q_significant_shortfall_count', 'q_near_conformant_count', 'q_major_surplus_count', 'curtailment_voltvar_count', 'nonconformance_voltvar_red_count', 'nonconformance_voltvar_red_alt_count', 'curtailment_eligible_count', 'null_uncurtailed_p_count', 'exposed_count', 'all_intervals_count', 'total_count', 'year', 'month']


In [ ]:
# =============================================================================
# Select a response mode and pull a ranked site list
# =============================================================================
# RESPONSE_MODE : "voltwatt" | "voltvar" | "sust_op" | "sust_op_3w" | "antiisland"
# BEHAVIOUR     : "nonconforming" | "conforming"
# PLOT_TYPE     : "operational" | "protective"
# YEAR / MONTH  : MONTH=None ranks over the full year
#
# All conformance tables live in solar_analytics_iceberg (SAI).
# Ranking metric per mode:
#   voltwatt   — nonconformance_voltwatt_count
#   voltvar    — nonconformance_voltvar_red_count  (severe Q deviation only)
#                extra breakdown columns also pulled for context
#   sust_op    — nonconformance_sust_op_count
#   sust_op_3w — nonconformance_sust_op_3w_count
#   antiisland — nonconformance_antiisland_count
#
# NOTE on voltvar category name swap (pipeline bug):
#   q_minor_deviation  = 10–90% band  (larger shortfall — counterintuitively named)
#   q_major_deficit    = 90–110% band (near-miss   — counterintuitively named)
# =============================================================================

RESPONSE_MODE = "voltvar"
BEHAVIOUR     = "nonconforming"   # "nonconforming" | "conforming"
PLOT_TYPE     = "operational"     # operational | protective
YEAR          = 2024
MONTH         = None              # None = full year; int = single month
MIN_DAYS      = 20
N_RESULTS     = 30

_MODE_MAP = {
    "voltwatt":   (SAI, "conformance_voltwatt",   "nonconformance_voltwatt_count"),
    "voltvar":    (SAI, "conformance_voltvar",     "nonconformance_voltvar_red_count"),
    "sust_op":    (SAI, "conformance_sust_op",     "nonconformance_sust_op_count"),
    "sust_op_3w": (SAI, "conformance_sust_op_3w",  "nonconformance_sust_op_3w_count"),
    "antiisland": (SAI, "conformance_antiisland",  "nonconformance_antiisland_count"),
}
if RESPONSE_MODE not in _MODE_MAP:
    raise ValueError(f"Unknown RESPONSE_MODE '{RESPONSE_MODE}'. "
                     f"Choose from: {list(_MODE_MAP)}")

_db, _table, _nonconf_col = _MODE_MAP[RESPONSE_MODE]
_order        = "DESC" if BEHAVIOUR == "nonconforming" else "ASC"
_year_filter  = f"year = {YEAR}"
_month_filter = f" AND month = {MONTH}" if MONTH is not None else ""
_period_label = f"{YEAR}-{MONTH:02d}" if MONTH else str(YEAR)

# voltvar only: pull full category breakdown for context in the ranked table
if RESPONSE_MODE == "voltvar":
    _extra_cols = """
        ,sum(q_adverse_count)           AS nc_adverse_intervals
        ,sum(q_inactive_count)          AS nc_inactive_intervals
        ,sum(q_minor_deviation_count)   AS nc_minor_deviation_intervals
        ,sum(q_major_deficit_count)     AS nc_major_deficit_intervals
        ,sum(q_major_surplus_count)     AS nc_major_surplus_intervals
        ,sum(curtailment_voltvar_count) AS nc_curtailment_intervals
        ,round(100.0 * sum(nonconformance_voltvar_count)
               / nullif(sum(total_count), 0), 2)  AS all_nc_pct
    """
else:
    _extra_cols = ""

ranked = aq(f"""
    SELECT
        site_id,
        sum({_nonconf_col})                                          AS nonconf_intervals,
        sum(total_count)                                             AS total_intervals,
        count(*)                                                     AS n_days,
        round(100.0 * sum({_nonconf_col})
              / nullif(sum(total_count), 0), 2)                      AS nonconf_pct
        {_extra_cols}
    FROM {_table}
    WHERE {_year_filter}{_month_filter}
    GROUP BY site_id
    HAVING sum(total_count) > 0
       AND count(*) >= {MIN_DAYS}
    ORDER BY nonconf_pct {_order}
    LIMIT {N_RESULTS}
""", database=_db)

print(f"{BEHAVIOUR.upper()} sites — {RESPONSE_MODE}  |  period: {_period_label}")
print(f"Database: {_db}  |  Table: {_table}")
print(f"Ranking column: {_nonconf_col}")
if RESPONSE_MODE == "voltvar":
    print("Extra columns: all_nc_pct = all NC / total_count (not just red category)")
    print("Category name swap: q_minor_deviation = big shortfall; "
          "q_major_deficit = near-miss")
print(f"Returned {len(ranked)} sites. Top = most {BEHAVIOUR}.")
ranked

NameError: name 'SAI' is not defined

In [ ]:
# =============================================================================
# Pick a site from the ranked list and pull its telemetry
# =============================================================================
# RANK_INDEX       : row in ranked (0 = top)
# OVERRIDE_SITE_ID : set to an int to bypass the ranked list
# FORCE_PLOT_MONTH : set to an int to override auto month selection
# FORCE_PLOT_YEAR  : set to an int to override YEAR (e.g. if site data is 2024
#                    but ranking was run on YEAR=2025)
# =============================================================================

RANK_INDEX       = 0
OVERRIDE_SITE_ID = 1600605164
FORCE_PLOT_MONTH = None   # e.g. 3 to force March
FORCE_PLOT_YEAR  = None   # e.g. 2024 to override YEAR

SELECTED_SITE_ID = (int(OVERRIDE_SITE_ID) if OVERRIDE_SITE_ID
                    else int(ranked["site_id"].iloc[RANK_INDEX]))
if SELECTED_SITE_ID in ranked["site_id"].values:
    print(f"Selected site_id = {SELECTED_SITE_ID}  "
          f"({ranked.loc[ranked.site_id==SELECTED_SITE_ID, 'nonconf_pct'].values[0]:.1f}% "
          f"non-conforming for {RESPONSE_MODE})")

# ── pull site metadata ────────────────────────────────────────────────────────
candidate = aq(f"""
    SELECT DISTINCT
        m.site_id, m.state, m.dnsp_name, m.ac_capacity_kw,
        m.manufacturer, m.model, m.min_time, m.max_time,
        c.circuit_id, c.circuit_polarity
    FROM meta_up23c m
    JOIN circuits c ON m.site_id = c.site_id
    WHERE c.is_pv = True
      AND m.site_id = {SELECTED_SITE_ID}
    LIMIT 1
""", database=SAI)

chosen_site_id     = int(candidate["site_id"].iloc[0])
chosen_circuit_id  = int(candidate["circuit_id"].iloc[0])
chosen_circuit_pol = int(candidate["circuit_polarity"].iloc[0])
chosen_mfr         = f"{candidate['manufacturer'].iloc[0]} {candidate['model'].iloc[0]}"
chosen_cap         = float(candidate["ac_capacity_kw"].iloc[0])

# Infer the year(s) this site actually has raw telemetry for, from metadata
_meta_min = pd.Timestamp(candidate["min_time"].iloc[0])
_meta_max = pd.Timestamp(candidate["max_time"].iloc[0])

print(f"\nSite ID:          {chosen_site_id}")
print(f"State:            {candidate['state'].iloc[0]}  |  DNSP: {candidate['dnsp_name'].iloc[0]}")
print(f"Capacity:         {chosen_cap:.1f} kW AC")
print(f"Inverter:         {chosen_mfr}")
print(f"Circuit polarity: {chosen_circuit_pol}  "
      f"({'export = negative power → flip sign' if chosen_circuit_pol == -1 else 'export = positive power → no flip'})")
print(f"Data from:        {_meta_min.date()} → {_meta_max.date()}")

# ── resolve PLOT_YEAR ─────────────────────────────────────────────────────────
# Use FORCE_PLOT_YEAR if set; otherwise use YEAR from the ranking cell,
# but warn and correct if the site's data doesn't cover that year.
if FORCE_PLOT_YEAR is not None:
    PLOT_YEAR = FORCE_PLOT_YEAR
    print(f"\nPLOT_YEAR forced to: {PLOT_YEAR}")
else:
    PLOT_YEAR = YEAR
    if PLOT_YEAR < _meta_min.year or PLOT_YEAR > _meta_max.year:
        PLOT_YEAR = _meta_max.year   # fall back to the last year with data
        print(f"\n[!]  YEAR={YEAR} is outside this site's data range "
              f"({_meta_min.year}–{_meta_max.year}). "
              f"Auto-correcting PLOT_YEAR → {PLOT_YEAR}.")
    else:
        print(f"\nPLOT_YEAR: {PLOT_YEAR}")

# ── resolve PLOT_MONTH ────────────────────────────────────────────────────────
# Cross-reference the conformance table (which months had most NC) against
# the months that actually have raw telemetry, so we never pick a ghost month.

if FORCE_PLOT_MONTH is not None:
    PLOT_MONTH = FORCE_PLOT_MONTH
    print(f"PLOT_MONTH forced to: {PLOT_MONTH}")

elif MONTH is not None:
    PLOT_MONTH = MONTH
    print(f"PLOT_MONTH inherited from ranking month: {PLOT_MONTH}")

else:
    _order_month = "DESC" if BEHAVIOUR == "nonconforming" else "ASC"

    # Step 1: which months have raw telemetry for this site + year?
    _telemetry_months = aq(f"""
        SELECT DISTINCT month
        FROM ts
        JOIN (SELECT DISTINCT circuit_id FROM meta_up23c
              WHERE is_pv = True AND site_id = {chosen_site_id}) m
          USING (circuit_id)
        WHERE year   = {PLOT_YEAR}
          AND is_pv  = True
        ORDER BY month
    """, database=SAI)

    _available_months = set(_telemetry_months["month"].tolist()) \
                        if not _telemetry_months.empty else set()
    print(f"\nMonths with raw telemetry in {PLOT_YEAR}: "
          f"{sorted(_available_months) if _available_months else 'none'}")

    if not _available_months:
        # No telemetry at all for PLOT_YEAR — this site is outside the data window
        print(f"[!]  No telemetry for {PLOT_YEAR}. "
              f"Set FORCE_PLOT_YEAR to a year between "
              f"{_meta_min.year} and {_meta_max.year}.")
        PLOT_MONTH = None

    else:
        # Step 2: get monthly NC ranking from the conformance table
        _monthly = aq(f"""
            SELECT
                month,
                sum({_nonconf_col})                                AS nc_intervals,
                sum(total_count)                                   AS total_intervals,
                round(100.0 * sum({_nonconf_col})
                      / nullif(sum(total_count), 0), 2)            AS nc_pct
            FROM {_table}
            WHERE year    = {PLOT_YEAR}
              AND site_id = {chosen_site_id}
            GROUP BY month
            HAVING sum(total_count) > 0
            ORDER BY nc_pct {_order_month}
        """, database=_db)

        print(f"\nMonthly NC breakdown for site {chosen_site_id} "
              f"(conformance table, {PLOT_YEAR}):")
        print(_monthly.to_string(index=False))

        # Step 3: pick the best NC month that also has raw telemetry
        _ranked_months = _monthly[
            _monthly["month"].isin(_available_months)
        ]

        if _ranked_months.empty:
            # Conformance table has no overlap with telemetry months
            # — just use the first available telemetry month
            PLOT_MONTH = sorted(_available_months)[0]
            print(f"\nNo overlap between conformance months and telemetry months. "
                  f"Falling back to first telemetry month: {PLOT_MONTH}")
        else:
            PLOT_MONTH = int(_ranked_months["month"].iloc[0])
            _best_nc   = float(_ranked_months["nc_pct"].iloc[0])
            print(f"\nAuto-selected PLOT_MONTH = {PLOT_MONTH}  "
                  f"({_best_nc:.1f}% NC — most {BEHAVIOUR} month "
                  f"with raw telemetry available)")

# pull telemetry
FIXED_OFFSET  = pytz.FixedOffset(600)
_month_clause = f"AND month = {PLOT_MONTH}" if PLOT_MONTH else ""

df = aq(f"""
    SELECT t_stamp, circuit_id, voltage, current, power, power_factor,
           energy_reactive
    FROM ts
    WHERE is_pv      = True
      AND year       = {PLOT_YEAR}
      {_month_clause}
      AND circuit_id = {chosen_circuit_id}
    ORDER BY t_stamp
""", database=SAI)

if df.empty:
    print(f"\n[!]  df is empty after pull (year={PLOT_YEAR}, month={PLOT_MONTH}, "
          f"circuit={chosen_circuit_id}). Cannot proceed.")
else:
    df["t_stamp_aest"] = (pd.to_datetime(df["t_stamp"])
                            .dt.tz_localize("UTC")
                            .dt.tz_convert(FIXED_OFFSET))
    df["P_kW"]   = df["power"]           / 1000 * chosen_circuit_pol
    df["Q_kvar"] = df["energy_reactive"] / 1000 * 12 * chosen_circuit_pol

    period = f"{PLOT_YEAR}" if not PLOT_MONTH else f"{PLOT_YEAR}-{PLOT_MONTH:02d}"
    print(f"\nRows pulled: {len(df):,}  "
          f"({df['t_stamp_aest'].min().date()} → {df['t_stamp_aest'].max().date()})")
    print(f"Period: {period}  ({'full year' if not PLOT_MONTH else 'single month'})")

    # ── smart day picker ──────────────────────────────────────────────────────
    _thresholds = {
        "voltwatt":   AS4777["VW"]["V1"],
        "voltvar":    AS4777["VVAR"]["V4"],
        "sust_op":    AS4777["SUSTOP_V_BAND"][1],
        "sust_op_3w": AS4777["SUSTOP_V_BAND"][1],
        "antiisland": AS4777["AI_V_BAND"][0],
    }
    _v_thr = _thresholds.get(RESPONSE_MODE, 253)

    days_with_events = (df[df["voltage"] > _v_thr]
                        .groupby(df["t_stamp_aest"].dt.date)["voltage"]
                        .count()
                        .sort_values(ascending=False))

    if days_with_events.empty:
        print(f"\nNo days where V > {_v_thr} V in {period}.")
        print(f"Voltage range in data: "
              f"{df['voltage'].min():.1f} – {df['voltage'].max():.1f} V")
        if RESPONSE_MODE == "voltvar":
            _v_thr_lo = AS4777["VVAR"]["V3"]
            days_with_events = (df[df["voltage"] > _v_thr_lo]
                                .groupby(df["t_stamp_aest"].dt.date)["voltage"]
                                .count()
                                .sort_values(ascending=False))
            if not days_with_events.empty:
                print(f"Falling back to V > {_v_thr_lo} V (V-VAr deadband hi):")
                print(days_with_events.head(10).to_string())
                print(f"\n  → Best candidate: {days_with_events.index[0]}"
                      f" ({days_with_events.iloc[0]} intervals above {_v_thr_lo} V)")
    else:
        print(f"\nDays where V > {_v_thr} V (most events first):")
        print(days_with_events.head(15).to_string())
        print(f"\n  → Best candidate: {days_with_events.index[0]}"
              f" ({days_with_events.iloc[0]} intervals above threshold)")
        print(f"  → Set OVERRIDE_DATE in the plot cell to one of these dates.")

In [ ]:
# Diagnostic: check what voltage the conformance table actually saw
# and compare to raw circuit-level max voltage

# 1. What voltage range does the conformance table record for this site?
conf_v = aq(f"""
    SELECT
        min(day)                    AS first_day,
        max(day)                    AS last_day,
        count(*)                    AS n_days,
        sum(nonconformance_sust_op_count)  AS total_nonconf,
        sum(total_count)            AS total_eligible
    FROM conformance_sust_op
    WHERE site_id = {SELECTED_SITE_ID}
      AND year = {PLOT_YEAR}
""", database=SAI)
print("Conformance table summary:")
display(conf_v)

# 2. How many circuits does this site have?
circuits_info = aq(f"""
    SELECT circuit_id, circuit_polarity, is_pv
    FROM meta_up23c
    WHERE site_id = {SELECTED_SITE_ID}
      AND is_pv = True
""", database=SAI)
print(f"\nPV circuits at this site ({len(circuits_info)}):")
display(circuits_info)

# 3. What is the max voltage across ALL circuits per timestamp?
# (This is what the conformance builder sees)
v_all_circuits = aq(f"""
    SELECT
        t_stamp,
        max(voltage) AS max_V_all_circuits,
        avg(voltage) AS avg_V_all_circuits,
        count(*)     AS n_circuits
    FROM ts
    JOIN (SELECT DISTINCT circuit_id
          FROM meta_up23c
          WHERE site_id = {SELECTED_SITE_ID} AND is_pv = True) c
      USING (circuit_id)
    WHERE year = {PLOT_YEAR}
      AND is_pv = True
    GROUP BY t_stamp
    HAVING max(voltage) > 255
    ORDER BY max_V_all_circuits DESC
    LIMIT 20
""", database=SAI)
print(f"\nTimestamps where max(V) across all circuits > 255 V:")
display(v_all_circuits)

## TZ Conversion

In [ ]:
# -----------------------------------------------------------------------------
# Timezone conversion: UTC → AEST (UTC+10, fixed offset, no daylight saving)
# Research convention from the CANVAS/CICCADA codebase: fixed offset avoids
# clock-change discontinuities in multi-month time-series analysis.
# UTC+10 means January plots show ~1hr behind real local clock. 
# Change to UTC+11 if you want calendar-accurate plots.
# -----------------------------------------------------------------------------

import pytz

FIXED_OFFSET = pytz.FixedOffset(600)   # 600 minutes = UTC+10 (AEST)

def to_aest(series):
    """Convert a UTC datetime series to AEST (UTC+10, fixed, no DST)."""
    if series.dt.tz is None:
        # Athena returns timezone-naive — tell pandas it's UTC first
        series = series.dt.tz_localize("UTC")
    return series.dt.tz_convert(FIXED_OFFSET)

df["t_stamp_aest"] = to_aest(df["t_stamp"])
print("Before:", df["t_stamp"].iloc[0])
print("After: ", df["t_stamp_aest"].iloc[0])

## Plotting functions

In [ ]:
# =============================================================================
# Plot functions
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.patches import Patch

def _strip_tz(series):
    """Remove timezone label without shifting values (required for fill_between)."""
    return series.dt.tz_localize(None)


### Operational Modes

In [ ]:
def plot_operational(df_day, site_id, ac_capacity_kw, zoom_date,
                     as4777, manufacturer="", figsize=(13, 13.5)):
    """
    5-panel unified daily plot for OPERATIONAL modes (Volt-Watt + Volt-VAr).

    Panel 0: Voltage: regime zones shaded, key setpoint lines labelled.
    Panel 1: Active power vs V-Watt ceiling (% S_rated left, kW right).
    Panel 2: Volt-Watt non-conformance (percentage points above ceiling).
    Panel 3: Reactive power vs required V-VAr band (% S_rated left, kvar right).
               Required band tracks instantaneous voltage — not flat limits.
    Panel 4: Volt-VAr non-conformance (signed pp outside band).
               Daily NC total annotated in kvarh and kvarh/kW nameplate.
    """


    vw  = as4777["VW"]
    vv  = as4777["VVAR"]
    tol = as4777["TOL_FRAC"]
    S   = ac_capacity_kw

    t = _strip_tz(df_day["t_stamp_aest"])
    V = df_day["voltage"]
    P = df_day["P_kW"]
    Q = df_day["Q_kvar"]

    # V-Watt ceiling as % S_rated (with +tol)
    def _vw_ceil_pct(v):
        if v < vw["V1"]: return 100.0
        if v > vw["V2"]: return vw["P2"] * 100.0
        return (1.0 - vw["P2"]) / (vw["V1"] - vw["V2"]) * (v - vw["V2"]) * 100.0 \
               + vw["P2"] * 100.0

    # V-VAr required Q: AS4777 Australia A 
    # Sign: + = supplying (lagging), − = absorbing (leading)
    def _vvar_required(v, s):
        if v <= vv["V1"]:   return  vv["Q1"] * s
        elif v <= vv["V2"]: return  vv["Q1"] * s * (vv["V2"] - v) / (vv["V2"] - vv["V1"])
        elif v <= vv["V3"]: return  0.0
        elif v <= vv["V4"]: return -vv["Q4"] * s * (v - vv["V3"]) / (vv["V4"] - vv["V3"])
        else:               return -vv["Q4"] * s

    # Volt-Watt: active power & ceiling in % S_rated
    P_pct      = (P / S) * 100.0
    P_ceil_pct = V.map(_vw_ceil_pct) + tol * 100.0
    P_nc_pct   = np.maximum(0.0, P_pct.values - P_ceil_pct.values)

    # Volt-VAr: required band in kvar and % S_rated
    Q_req         = V.map(lambda v: _vvar_required(v, S))
    Q_req_max     = Q_req + tol * S
    Q_req_min     = Q_req - tol * S
    Q_pct         = (Q         / S) * 100.0
    Q_req_pct     = (Q_req     / S) * 100.0
    Q_req_max_pct = (Q_req_max / S) * 100.0
    Q_req_min_pct = (Q_req_min / S) * 100.0

    # V-VAr NC: signed pp for bars, kvar magnitude for energy totals
    nc_vvar_kvar  = np.maximum(0, Q - Q_req_max) + np.maximum(0, Q_req_min - Q)
    nc_signed_pct = np.where(
        Q_pct.values > Q_req_max_pct.values,
         Q_pct.values - Q_req_max_pct.values,
        np.where(
            Q_pct.values < Q_req_min_pct.values,
            -(Q_req_min_pct.values - Q_pct.values),
            0.0))

    INTERVAL_H      = 5 / 60
    nc_kvarh        = nc_vvar_kvar.sum() * INTERVAL_H
    nc_kvarh_per_kw = nc_kvarh / S

    # zone flags
    vvar_active = V.values >  vv["V3"]   # V > 240 V-VAr required
    vw_active   = V.values >= vw["V1"]   # V ≥ 253 V-Watt active

    # colours
    Cv    = "#b45309"   # amber  — voltage
    Cp    = "#2e7d32"   # green  — active power
    Cc    = "#1a1a1a"   # black  — ceiling line
    Cq    = "#1565c0"   # blue   — reactive power
    Cn    = "#c62828"   # red    — non-conformance
    C_REF = "#f59e0b"   # yellow — required band
    C_VVAR = "#7c3aed"  # violet — V-VAr zone
    C_VW   = "#4709b2"  # indigo — V-Watt zone (darker violet)
    C_GRID = "#ebebeb"

    # layout: V | P | VW-NC | Q | VVAr-NC
    fig, axes = plt.subplots(
        5, 1, figsize=figsize, dpi=130, sharex=True,
        gridspec_kw={"height_ratios": [1.8, 2.2, 1.0, 2.2, 1.0]})
    fig.subplots_adjust(hspace=0.05, left=0.10, right=0.90, top=0.95, bottom=0.05)
    ax_v, ax_p, ax_pnc, ax_q, ax_qnc = axes

    # shared graduated zone shading: V-VAr zone lighter, V-Watt zone darker
    # where both are active the colours blend, making the 253–258 V overlap visible
    for ax in axes:
        ax.fill_between(t, 0, 1, where=vvar_active,
                        transform=ax.get_xaxis_transform(),
                        color=C_VVAR, alpha=0.07, linewidth=0, zorder=0)
        ax.fill_between(t, 0, 1, where=vw_active,
                        transform=ax.get_xaxis_transform(),
                        color=C_VW, alpha=0.08, linewidth=0, zorder=0)

    _vvar_patch = Patch(color=C_VVAR, alpha=0.30,
                        label=f"V > {vv['V3']:.0f} V — V-VAr required")
    _vw_patch   = Patch(color=C_VW,   alpha=0.30,
                        label=f"V ≥ {vw['V1']:.0f} V — V-Watt active")

    # ═══ PANEL 0 Voltage ═══════════════════════════════════════════════
    ax_v.plot(t, V, color=Cv, lw=1.3, zorder=4)
    for vref, ls, al, lbl in [
        (vv["V3"], ":",  0.55, "240 V (V-VAr deadband hi)"),
        (vw["V1"], "--", 0.85, "253 V (V-Watt start / V-VAr hi ramp)"),
        (vv["V4"], "--", 0.85, "258 V (V-VAr max absorb)"),
        (vw["V2"], "--", 0.85, "260 V (V-Watt full curtail)"),
    ]:
        ax_v.axhline(vref, color=Cv, lw=0.8, ls=ls, alpha=al, zorder=3)
        ax_v.text(t.iloc[-1], vref + 0.25, lbl, va="bottom", ha="right",
                  fontsize=6, color=Cv, alpha=min(al + 0.15, 1.0))
    ax_v.set_ylabel("Voltage (V)", fontsize=8.5, color=Cv)
    ax_v.tick_params(axis="y", colors=Cv, labelsize=8)
    ax_v.set_ylim(min(228, V.min() - 2), max(262, V.max() + 2))
    ax_v.grid(color=C_GRID, lw=0.5); ax_v.set_facecolor("white")
    ax_v.legend(handles=[_vvar_patch, _vw_patch], fontsize=7,
                loc="upper left", framealpha=0.92, edgecolor="#cccccc", ncol=1)
    plt.setp(ax_v.get_xticklabels(), visible=False)

    # ═══ PANEL 1 Active power (% left, kW right) ═══════════════════════
    ax_p.plot(t, P_ceil_pct, color=Cc, lw=1.6, zorder=4,
              label=f"V-Watt ceiling (+{tol*100:.0f}% tol, % S_rated)")
    ax_p.plot(t, P_pct, color=Cp, lw=1.3, zorder=5,
              label="Measured P (% S_rated)")
    ax_p.axhline(100, color=Cp, lw=0.6, ls=":", alpha=0.45, zorder=3,
                 label="100% S_rated (nameplate)")
    ax_p.axhline(vw["P2"] * 100, color=Cc, lw=0.6, ls=":", alpha=0.45, zorder=3,
                 label=f"V-Watt floor: {vw['P2']*100:.0f}% at ≥{vw['V2']:.0f} V")
    ax_p.set_ylabel("Active power\n(% S_rated)", fontsize=8.5, color=Cp)
    ax_p.tick_params(axis="y", colors=Cp, labelsize=8)
    P_LO, P_HI = -2, 115
    ax_p.set_ylim(P_LO, P_HI); ax_p.set_yticks([0, 20, 40, 60, 80, 100])
    ax_p.grid(color=C_GRID, lw=0.5); ax_p.set_facecolor("white")
    ax_p.legend(fontsize=7.5, loc="upper left", framealpha=0.9)
    plt.setp(ax_p.get_xticklabels(), visible=False)
    # kW secondary axis — pure linear rescale, no data re-plotted
    ax_pkw = ax_p.twinx()
    ax_pkw.set_ylim(P_LO / 100 * S, P_HI / 100 * S)
    ax_pkw.set_ylabel("Active power\n(kW)", fontsize=8.5, color=Cp)
    ax_pkw.tick_params(axis="y", colors=Cp, labelsize=8)
    _kw = np.linspace(0, S, 6)
    ax_pkw.set_yticks(_kw); ax_pkw.set_yticklabels([f"{v:.1f}" for v in _kw])

    # ═══ PANEL 2 Volt-Watt NC (pp above ceiling) ═══════════════════════
    ax_pnc.bar(t, P_nc_pct, width=pd.Timedelta(minutes=4.5),
               color=Cn, alpha=0.80, align="center", zorder=4,
               label="V-Watt NC (pp above ceiling)")
    ax_pnc.axhline(0, color="k", lw=0.5, zorder=3)
    ax_pnc.set_ylabel("V-W NC\n(pp)", fontsize=8.5, color=Cn)
    ax_pnc.tick_params(axis="y", colors=Cn, labelsize=8)
    ax_pnc.set_ylim(0, max(P_nc_pct.max() * 1.15, 2.0))
    ax_pnc.grid(color=C_GRID, lw=0.5, axis="y"); ax_pnc.set_facecolor("white")
    ax_pnc.legend(fontsize=7.5, loc="upper left", framealpha=0.9)
    plt.setp(ax_pnc.get_xticklabels(), visible=False)

    # ═══ PANEL 3 Reactive power vs band (% left, kvar right) ═══════════
    ax_q.fill_between(t, Q_req_min_pct, Q_req_max_pct,
                      color=C_REF, alpha=0.25, linewidth=0, zorder=1)
    ax_q.plot(t, Q_req_min_pct, color=C_REF, lw=0.8, ls="--", zorder=2)
    ax_q.plot(t, Q_req_max_pct, color=C_REF, lw=0.8, ls="--", zorder=2)
    ax_q.plot(t, Q_req_pct,     color=C_REF, lw=1.0, ls="-",  alpha=0.6, zorder=2)
    ax_q.plot(t, Q_pct,         color=Cq,    lw=1.4, zorder=4)
    ax_q.axhline(0, color="k", lw=0.5, zorder=3)
    ax_q.set_ylabel("Reactive power\n(% S_rated, + sup / - abs)", fontsize=8.5, color=Cq)
    ax_q.tick_params(axis="y", colors=Cq, labelsize=8)
    _qlim = max(vv["Q1"], vv["Q4"]) * 100 + 10
    ax_q.set_ylim(-_qlim, _qlim)
    ax_q.grid(color=C_GRID, lw=0.5); ax_q.set_facecolor("white")
    ax_q.legend(handles=[
        Patch(color=C_REF, alpha=0.40,
              label=f"Required Q band (±{tol*100:.0f}% S_rated)"),
        plt.Line2D([0], [0], color=Cq, lw=1.4,
                   label="Measured Q (% S_rated)"),
    ], fontsize=7.5, loc="upper left", framealpha=0.9)
    plt.setp(ax_q.get_xticklabels(), visible=False)
    # kvar secondary axis
    ax_qkvar = ax_q.twinx()
    ax_qkvar.set_ylim(-_qlim / 100 * S, _qlim / 100 * S)
    ax_qkvar.set_ylabel("Reactive power\n(kvar)", fontsize=8.5, color=Cq)
    ax_qkvar.tick_params(axis="y", colors=Cq, labelsize=8)

    # ═══ PANEL 4 Volt-VAr NC (signed pp) ═══════════════════════════════
    ax_qnc.bar(t, nc_signed_pct, width=pd.Timedelta(minutes=4.5),
               color=Cn, alpha=0.80, align="center", zorder=4)
    ax_qnc.axhline(0, color="k", lw=0.5, zorder=3)
    ax_qnc.set_ylabel("V-VAr NC\n(pp, signed)", fontsize=8.5, color=Cn)
    ax_qnc.tick_params(axis="y", colors=Cn, labelsize=8)
    ax_qnc.grid(color=C_GRID, lw=0.5, axis="y"); ax_qnc.set_facecolor("white")
    ax_qnc.legend(handles=[
        plt.Line2D([0], [0], color=Cn, lw=4, alpha=0.80,
                   label="V-VAr NC (pp outside band, signed)"),
    ], fontsize=7.5, loc="upper left", framealpha=0.9)
    ax_qnc.text(0.99, 0.95,
                f"Daily V-VAr NC:  {nc_kvarh:.3f} kvarh"
                f"   ({nc_kvarh_per_kw:.4f} kvarh / kW nameplate)",
                transform=ax_qnc.transAxes, ha="right", va="top",
                fontsize=7.5, color=Cn,
                bbox=dict(boxstyle="round,pad=0.3", fc="white",
                          ec=Cn, alpha=0.85, lw=0.7))
    ax_qnc.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax_qnc.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    ax_qnc.xaxis.set_minor_locator(mdates.HourLocator(interval=1))
    ax_qnc.tick_params(axis="x", which="major", labelsize=8)
    fig.autofmt_xdate(rotation=0, ha="center")

    fig.suptitle(
        f"Site {site_id}  ·  {manufacturer}  ·  {zoom_date}\n"
        f"Operational modes: Volt-Watt & Volt-VAr  ·  Nameplate {S:.0f} kW AC",
        fontsize=10, fontweight="bold", y=0.975)
    plt.show()

    # summary printout
    n_vw_nc   = int((P_nc_pct > 0).sum())
    n_vvar_nc = int((nc_vvar_kvar > 0).sum())
    print(f"\nVolt-Watt:  {n_vw_nc} non-conforming intervals")
    print(f"Volt-VAr:   {n_vvar_nc} non-conforming intervals  |  "
          f"{nc_kvarh:.3f} kvarh  ({nc_kvarh_per_kw:.4f} kvarh/kW)")


print("Functions registered: plot_operational(), plot_protective()")
print("Set zoom_date and run the next cell to plot.")

### Protective Functions

In [ ]:
def plot_protective(df_day, site_id, ac_capacity_kw, zoom_date,
                    as4777, manufacturer="", figsize=(12, 7)):
    """
    2-panel daily plot for PROTECTIVE functions (Sustained-Op + Anti-Islanding).

    Panel 1 — Voltage:
        Zoomed to observed range. OV thresholds (258, 265, 275 V) shown as
        reference lines with left-spine labels and zone shading.
        UV thresholds (180, 70 V) shown as a text annotation only — they sit far
        below normal operating range and would compress the y-axis uselessly.

    Panel 2 — Active power P only:
        The conformance criterion is whether P drops below 4% of nameplate
        within the required time after the voltage threshold is exceeded.
        Reactive power is not a compliance criterion for these modes.
    """
    import matplotlib.transforms as mtransforms
    from matplotlib.patches import Patch

    t   = _strip_tz(df_day["t_stamp_aest"])
    V   = df_day["voltage"]
    P   = df_day["P_kW"]
    S   = ac_capacity_kw

    so  = as4777["SUSTOP_V_CEIL"]    # 258 V
    ov1 = as4777["AI_OV1"]           # 265 V
    ov2 = as4777["AI_OV2"]           # 275 V
    uv1 = as4777.get("AI_UV1", 180)
    uv2 = as4777.get("AI_UV2",  70)

    C_VOLT = "#b45309"; C_P = "#2e7d32"; C_GRID = "#ebebeb"
    C_SO   = "#f59e0b"
    C_OV   = "#c62828"
    C_UV   = "#1565c0"

    fig = plt.figure(figsize=figsize, dpi=140)
    fig.subplots_adjust(left=0.13, right=0.97, top=0.93, bottom=0.08)
    gs   = fig.add_gridspec(2, 1, height_ratios=[1.6, 2.0], hspace=0.06)
    ax_v = fig.add_subplot(gs[0])
    ax_p = fig.add_subplot(gs[1], sharex=ax_v)

    # ── OV zone shading ──────────────────────────────────────────────────
    for ax in (ax_v, ax_p):
        ax.fill_between(t, 0, 1,
                        where=(V.values >= so) & (V.values < ov1),
                        transform=ax.get_xaxis_transform(),
                        color=C_SO, alpha=0.18, linewidth=0, zorder=0)
        ax.fill_between(t, 0, 1,
                        where=(V.values >= ov1) & (V.values < ov2),
                        transform=ax.get_xaxis_transform(),
                        color=C_OV, alpha=0.18, linewidth=0, zorder=0)
        ax.fill_between(t, 0, 1,
                        where=V.values >= ov2,
                        transform=ax.get_xaxis_transform(),
                        color=C_OV, alpha=0.35, linewidth=0, zorder=0)

    # ═══ PANEL 1 — Voltage ═══════════════════════════════════════════════
    ax_v.plot(t, V, color=C_VOLT, lw=1.4, zorder=4)
    ax_v.fill_between(t, so, V, where=V.values >= so,
                      color=C_VOLT, alpha=0.18, linewidth=0, zorder=2)

    v_lo = max(V.min() - 3, 230)
    v_hi = max(V.max() + 5, ov2 + 3)
    ax_v.set_ylim(v_lo, v_hi)

    OV_REFS = [
        (so,  C_SO,      "-",  1.2, 0.90, "bold",   f"{so:.0f} V  Sust-Op limit"),
        (ov1, C_OV,      "--", 1.0, 0.85, "bold",   f"{ov1:.0f} V  OV1 (trip ≤2 s)"),
        (ov2, "#7f1d1d", ":",  0.8, 0.65, "normal", f"{ov2:.0f} V  OV2 (trip ≤0.2 s)"),
    ]
    for vref, col, ls, lw_, al, fw, _ in OV_REFS:
        if v_lo <= vref <= v_hi:
            ax_v.axhline(vref, color=col, lw=lw_, ls=ls, alpha=al, zorder=3)

    fig.canvas.draw()
    blend = mtransforms.blended_transform_factory(ax_v.transAxes, ax_v.transData)
    for vref, col, _, _, al, fw, lbl in OV_REFS:
        if v_lo <= vref <= v_hi:
            ax_v.annotate("", xy=(0, vref), xycoords=blend,
                xytext=(-0.055, vref), textcoords=blend,
                arrowprops=dict(arrowstyle="-", color=col, lw=0.9,
                                alpha=al, shrinkA=0, shrinkB=0),
                zorder=7, clip_on=False)
            ax_v.text(-0.06, vref, lbl, transform=blend,
                      ha="right", va="center", fontsize=7.5,
                      color=col, fontweight=fw, alpha=al, clip_on=False)

    # UV annotation — zorder=6 ensures it sits above the voltage line (zorder=4)
    ax_v.text(0.99, 0.04,
              f"Under-voltage thresholds (below observable range):\n"
              f"  UV1: V ≤ {uv1:.0f} V — trip ≤11 s\n"
              f"  UV2: V ≤ {uv2:.0f} V — trip ≤2 s",
              transform=ax_v.transAxes, ha="right", va="bottom",
              fontsize=6.5, color=C_UV, zorder=6,
              bbox=dict(boxstyle="round,pad=0.3", fc="white",
                        ec=C_UV, alpha=0.90, lw=0.7))

    visible_ticks = [v for v in [230, 240, 250, 258, 260, 265, 270, 275, 280]
                     if v_lo <= v <= v_hi]
    ax_v.set_yticks(visible_ticks)
    ax_v.set_yticklabels([str(v) for v in visible_ticks])
    ax_v.set_ylabel("Voltage (V)", fontsize=8.5, color=C_VOLT)
    ax_v.tick_params(axis="y", colors=C_VOLT, labelsize=8)
    ax_v.grid(True, color=C_GRID, lw=0.5, zorder=0)
    ax_v.set_facecolor("white")
    plt.setp(ax_v.get_xticklabels(), visible=False)
    ax_v.legend(handles=[
        Patch(color=C_SO, alpha=0.50,
              label=f"Sust-Op zone ≥{so:.0f} V  (cease in ≤15 min)"),
        Patch(color=C_OV, alpha=0.55,
              label=f"Anti-island OV1 ≥{ov1:.0f} V  (cease in ≤2 s)"),
    ], fontsize=7.5, loc="upper left", framealpha=0.92, edgecolor="#cccccc")

    # ═══ PANEL 2 — Active power ══════════════════════════════════════════
    # Conformance criterion: did P drop below 4% nameplate after the
    # voltage threshold was exceeded? Reactive power is not assessed here.
    ax_p.plot(t, P, color=C_P, lw=1.6, zorder=4,
              label="Active power P (kW)")
    ax_p.axhline(S,      color=C_P, lw=0.7, ls=":",  alpha=0.45, zorder=3,
                 label=f"Nameplate {S:.0f} kW")
    ax_p.axhline(S*0.04, color=C_P, lw=0.8, ls="--", alpha=0.65, zorder=3,
                 label=f"4% floor ({S*0.04:.2f} kW) — conformance threshold")
    ax_p.axhline(0, color="k", lw=0.5, zorder=3)
    ax_p.set_ylabel("Active power (kW)", fontsize=8.5, color=C_P)
    ax_p.tick_params(axis="y", colors=C_P, labelsize=8)
    ax_p.set_ylim(bottom=-0.3)
    ax_p.legend(fontsize=7.5, loc="upper left", framealpha=0.92, edgecolor="#cccccc")
    ax_p.grid(True, color=C_GRID, lw=0.5, zorder=0)
    ax_p.set_facecolor("white")

    ax_p.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax_p.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    ax_p.xaxis.set_minor_locator(mdates.HourLocator(interval=1))
    ax_p.tick_params(axis="x", which="major", labelsize=8, pad=3)
    ax_p.tick_params(axis="x", which="minor", length=2)
    fig.autofmt_xdate(rotation=0, ha="center")

    ax_v.set_title(
        f"Site {site_id}  ·  {manufacturer}  ·  {zoom_date}\n"
        f"Protective functions: Sustained-Op & Anti-Islanding  ·  Nameplate {S:.0f} kW",
        fontsize=10, fontweight="bold", pad=6, loc="left")
    plt.show()

## Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from matplotlib.patches import Patch

In [ ]:
# =============================================================================
# Pick a day and plot
# =============================================================================
# OPTION A — auto-pick the best high-voltage day: leave OVERRIDE_DATE = None.
#   Full-year mode: picks the day with most intervals above the mode threshold.
#   Single-month mode: defaults to the 7th of PLOT_MONTH.
#
# OPTION B — pin a specific date: set OVERRIDE_DATE = "YYYY-MM-DD".
#
# PLOT_TYPE ("operational" / "protective") is inherited from the config cell.
# =============================================================================

import matplotlib.pyplot as plt

OVERRIDE_DATE = None          # e.g. "2024-02-08"  ← change to pin a date

ELSE_DAY_PLOT = 27

if OVERRIDE_DATE is not None:
    zoom_date = OVERRIDE_DATE
    print(f"Using manually specified date: {zoom_date}")
elif PLOT_MONTH is None:
    zoom_date = str(days_with_events.index[0]) if not days_with_events.empty \
                else f"{PLOT_YEAR}-01-01"
    print(f"Full-year mode — auto-selected: {zoom_date}")
else:
    zoom_date = f"{PLOT_YEAR}-{PLOT_MONTH:02d}-{ELSE_DAY_PLOT}"
    print(f"Single-month mode — using: {zoom_date}")

mask   = df["t_stamp_aest"].dt.date == pd.Timestamp(zoom_date).date()
df_day = df[mask].copy()

if df_day.empty:
    print(f"No data for {zoom_date}.")
    if df.empty:
        print("df itself is empty. Re-run the site-picker cell.")
    else:
        _available = sorted(df["t_stamp_aest"].dt.date.unique())
        print(f"df covers {_available[0]} → {_available[-1]}  "
              f"({len(_available)} days total)")
        print(f"First 10 dates: {_available[:10]}")
        print(f"Try setting OVERRIDE_DATE to one of the dates above, "
              f"or set FORCE_PLOT_MONTH in the site-picker to a month "
              f"that exists in this range.")
else:
    if "Q_kvar" not in df_day.columns and "energy_reactive" in df_day.columns:
        df_day["Q_kvar"] = df_day["energy_reactive"] / 1000 * 12

    plot_kwargs = dict(
        df_day         = df_day,
        site_id        = chosen_site_id,
        ac_capacity_kw = chosen_cap,
        zoom_date      = zoom_date,
        as4777         = AS4777,
        manufacturer   = chosen_mfr,
    )

    if PLOT_TYPE == "operational":
        print("Plotting operational modes (Volt-Watt + Volt-VAr)...")
        plot_operational(**plot_kwargs)
    elif PLOT_TYPE == "protective":
        print("Plotting protective modes (Sustained-Op & Anti-Islanding)...")
        plot_protective(**plot_kwargs)
    else:
        print(f"Unknown PLOT_TYPE '{PLOT_TYPE}'. Use 'operational' or 'protective'.")

In [ ]:
# ── 3.3c  Volt-VAr monthly scatter: Q vs V for one site-month ────────────────
# Each dot is one 5-minute interval. The required Q curve and ±4% tolerance
# band are overlaid so you can see how well the inverter tracks the standard
# across the full voltage distribution for the month.
#
# Month is derived from zoom_date (set in the plot cell) so it always matches
# a month with data.
# Plot 1: overview — Y-axis fixed ±100% S_rated; X-axis fixed 200–280 V.
# Plot 2: zoomed  — Y-axis −60% to +40%; X-axis 230–260 V (operating range).
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np

vv  = AS4777["VVAR"]
tol = AS4777["TOL_FRAC"]

# ── V-VAr required Q: self-contained definition so this cell can run
# independently of the plot cell. Same piecewise curve as everywhere else.
# Sign: + = supplying (lagging), − = absorbing (leading).
def _vvar_required(v, s):
    if v <= vv["V1"]:   return  vv["Q1"] * s
    elif v <= vv["V2"]: return  vv["Q1"] * s * (vv["V2"] - v) / (vv["V2"] - vv["V1"])
    elif v <= vv["V3"]: return  0.0
    elif v <= vv["V4"]: return -vv["Q4"] * s * (v - vv["V3"]) / (vv["V4"] - vv["V3"])
    else:               return -vv["Q4"] * s

# ── derive month from zoom_date ───────────────────────────────────────────────
if "zoom_date" not in dir() or not zoom_date:
    raise ValueError("zoom_date is not set. Run the plot cell first.")

_scatter_ts    = pd.Timestamp(zoom_date)
_scatter_year  = _scatter_ts.year
_scatter_month = _scatter_ts.month
_period_label  = f"{_scatter_year}-{_scatter_month:02d}"

print(f"Pulling V-VAr scatter data for site {chosen_site_id}, {_period_label} ...")

# ── diagnostic: check what data exists for this site ─────────────────────────
_check = aq(f"""
    SELECT year, month, count(*) AS n_rows
    FROM ts
    JOIN (
        SELECT DISTINCT circuit_id
        FROM meta_up23c
        WHERE is_pv = True AND site_id = {chosen_site_id}
    ) m USING (circuit_id)
    WHERE year = {_scatter_year}
      AND is_pv = True
    GROUP BY year, month
    ORDER BY year, month
""", database=SAI)
print("Data availability for this site:")
print(_check.to_string(index=False))

# ── pull per-interval site-level aggregates ───────────────────────────────────
scatter_df = aq(f"""
    WITH site_agg AS (
        SELECT
            t_stamp,
            sum(power            * m.circuit_polarity / 1000)      AS P_kW,
            sum(energy_reactive  * m.circuit_polarity / 1000 * 12) AS Q_kvar,
            avg(voltage)                                            AS V
        FROM ts
        JOIN (
            SELECT DISTINCT circuit_id, circuit_polarity
            FROM meta_up23c
            WHERE is_pv   = True
              AND site_id = {chosen_site_id}
        ) m USING (circuit_id)
        WHERE year    = {_scatter_year}
          AND month   = {_scatter_month}
          AND is_pv   = True
          AND voltage > 0
          AND voltage < 300
        GROUP BY t_stamp
    )
    SELECT t_stamp, P_kW, Q_kvar, V
    FROM site_agg
    ORDER BY t_stamp
""", database=SAI)

print(f"Pulled {len(scatter_df):,} intervals.")

if scatter_df.empty:
    print("Still no data. Check the availability table printed above and "
          "set zoom_date to a date in a month shown there.")
else:
    S = chosen_cap

    # ── required Q curve on a fine grid covering both plots ───────────────
    V_grid        = np.linspace(200, 280, 800)
    Q_req_abs     = np.array([_vvar_required(v, S) for v in V_grid])
    Q_req_pct     = (Q_req_abs / S) * 100.0
    Q_req_max_pct = Q_req_pct + tol * 100.0
    Q_req_min_pct = Q_req_pct - tol * 100.0

    # ── per-interval conformance flag ─────────────────────────────────────
    scatter_df["Q_req"]     = scatter_df["V"].map(lambda v: _vvar_required(v, S))
    scatter_df["Q_req_max"] = scatter_df["Q_req"] + tol * S
    scatter_df["Q_req_min"] = scatter_df["Q_req"] - tol * S
    scatter_df["Q_pct"]     = (scatter_df["Q_kvar"] / S) * 100.0
    scatter_df["nc"] = (
        (scatter_df["Q_kvar"] > scatter_df["Q_req_max"]) |
        (scatter_df["Q_kvar"] < scatter_df["Q_req_min"])
    )

    n_total = len(scatter_df)
    n_nc    = int(scatter_df["nc"].sum())
    pct_nc  = 100.0 * n_nc / n_total if n_total > 0 else 0.0

    C_OK  = "#1565c0"
    C_NC  = "#c62828"
    C_REF = "#f59e0b"
    ok_mask = ~scatter_df["nc"]

    def _draw_scatter(ax, x_lo, x_hi, y_lo, y_hi, x_step, y_step, subtitle):
        ax.scatter(scatter_df.loc[ok_mask,  "V"],
                   scatter_df.loc[ok_mask,  "Q_pct"],
                   s=4, alpha=0.20, color=C_OK, zorder=3,
                   label=f"Conforming ({ok_mask.sum():,} intervals)")
        ax.scatter(scatter_df.loc[~ok_mask, "V"],
                   scatter_df.loc[~ok_mask, "Q_pct"],
                   s=6, alpha=0.45, color=C_NC, zorder=4,
                   label=f"Non-conforming ({n_nc:,} intervals,  {pct_nc:.1f}%)")
        ax.plot(V_grid, Q_req_pct, color=C_REF, lw=1.8, zorder=5,
                label="Required Q (AS4777 curve)")
        ax.fill_between(V_grid, Q_req_min_pct, Q_req_max_pct,
                        color=C_REF, alpha=0.20, linewidth=0, zorder=2,
                        label=f"±{tol*100:.0f}% S_rated tolerance band")
        ax.axhline(0, color="k", lw=0.5, zorder=1)
        for vx, lbl in [
            (vv["V1"], f"V1 {vv['V1']:.0f} V"),
            (vv["V2"], f"V2 {vv['V2']:.0f} V"),
            (vv["V3"], f"V3 {vv['V3']:.0f} V"),
            (vv["V4"], f"V4 {vv['V4']:.0f} V"),
        ]:
            if x_lo <= vx <= x_hi:
                ax.axvline(vx, color="grey", lw=0.6, ls=":", zorder=1)
                ax.text(vx + 0.4, y_hi * 0.95, lbl, fontsize=6,
                        color="grey", va="top", ha="left")
        ax.set_xlim(x_lo, x_hi)
        ax.set_ylim(y_lo, y_hi)
        ax.set_xticks(range(x_lo, x_hi + 1, x_step))
        ax.set_yticks(range(y_lo, y_hi + 1, y_step))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:+.0f}%"))
        ax.set_xlabel("Voltage (V)", fontsize=9)
        ax.set_ylabel("Reactive power (% S_rated)\n+ = supplying,  − = absorbing",
                      fontsize=9)
        ax.set_title(
            f"Site {chosen_site_id}  ·  {chosen_mfr}  ·  {_period_label}  ·  {subtitle}\n"
            f"Volt-VAr response vs AS/NZS 4777.2:2020  ·  Nameplate {S:.0f} kW AC",
            fontsize=9, fontweight="bold", loc="left")
        ax.legend(fontsize=7.5, loc="lower left", framealpha=0.92, edgecolor="#cccccc")
        ax.grid(color="#ebebeb", lw=0.5)
        ax.set_facecolor("white")

    # ═══ PLOT 1 — full overview (200–280 V, ±100%) ════════════════════════
    fig1, ax1 = plt.subplots(figsize=(9, 5.5), dpi=130)
    _draw_scatter(ax1, x_lo=200, x_hi=280, y_lo=-100, y_hi=100,
                  x_step=5, y_step=20, subtitle="overview")
    plt.tight_layout()
    plt.show()

    # ═══ PLOT 2 — operating range zoom (230–260 V, −60% to +40%) ═════════
    fig2, ax2 = plt.subplots(figsize=(9, 5.5), dpi=130)
    _draw_scatter(ax2, x_lo=230, x_hi=260, y_lo=-60, y_hi=40,
                  x_step=2, y_step=10, subtitle="operating range zoom")
    plt.tight_layout()
    plt.show()

    print(f"Non-conforming: {n_nc:,} / {n_total:,} intervals  ({pct_nc:.1f}%)")